# COMP8420 Assignment 3 — Use Case 1: Intelligent Customer Service Systems

## P5 — System Integration and UI Development

**Student Name:** Joel Kuzhikalavayalil James

**Student ID:** 49019457

---

### Overview

This notebook implements the final integration stage of the Intelligent Customer Service System developed for COMP8420 Assignment 3. The primary objective of P5 is to integrate all components developed throughout the project into a complete end-to-end customer support solution.

The integrated system combines the following project components:

* **P2:** Basic Natural Language Processing (NLP) analysis, including sentiment analysis, category classification, part-of-speech (POS) tagging, and named entity recognition (NER).
* **P4:** Retrieval-Augmented Generation (RAG) and agent-based decision-making for policy retrieval and escalation handling.
* **P3:** Qwen 2.5-1.5B-Instruct enhanced with LoRA fine-tuning for customer support response generation.
* **P5:** System integration, Streamlit user interface development, database logging, pipeline testing, and deployment.

### System Workflow

The final integrated system enables a customer to submit a support request through a web-based interface. The message is processed through multiple stages:

1. NLP analysis extracts important information such as customer details, products, dates, issues, and sentiment.
2. The RAG component retrieves relevant company policies and support guidelines.
3. The agent module determines the appropriate action, including whether the case requires escalation to a human support specialist.
4. The fine-tuned language model generates a professional customer support response.
5. The Streamlit interface presents the generated response along with real-time analytics, extracted entities, sentiment, category classification, and retrieved policy information.

This integration demonstrates how modern NLP, Retrieval-Augmented Generation, agent-based reasoning, and large language models can be combined to build an intelligent customer service system capable of assisting users in realistic support scenarios.


## 1 Install Dependencies and Import Libraries

This section installs all required packages and imports the libraries used throughout the notebook. The installation step only needs to be executed once. If the dependencies are already installed in the environment, this section can be skipped.

In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
!pip uninstall -y transformers peft sentence-transformers bitsandbytes torchao -q

!pip install -q transformers==4.45.2
!pip install -q peft==0.19.1
!pip install -q sentence-transformers==3.1.1
!pip install -q sentencepiece
!pip install -q bitsandbytes
!pip install -q accelerate faiss-cpu datasets scikit-learn joblib spacy pandas numpy matplotlib seaborn nbformat streamlit

!python -m spacy download en_core_web_sm -q

print("All dependencies installed successfully")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
All dependencies installed successfully


In [3]:
# ==========================================
# Core Imports
# ==========================================

import os
import re
import json
import pickle
from datetime import datetime

# ==========================================
# Data / NLP
# ==========================================

import numpy as np
import pandas as pd

# ==========================================
# Streamlit UI
# ==========================================

import streamlit as st

# ==========================================
# Machine Learning
# ==========================================

from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# Hugging Face / LLM
# ==========================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import (
    PeftModel,
    PeftConfig
)

# ==========================================
# Sentence Embeddings for RAG
# ==========================================

from sentence_transformers import SentenceTransformer

# ==========================================
# SpaCy NLP
# ==========================================

import spacy

# ==========================================
# Optional
# ==========================================

import warnings
warnings.filterwarnings("ignore")

c:\Users\joelk\anaconda3\envs\comp8420\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. GPU Check

In [4]:
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## 3.Working Directory Setup

This section configures the notebook working directory and verifies that all project files are available. It ensures that the Python path includes the project folder and displays the notebooks and supporting files required for system integration.

In [5]:
import sys, os

# Local folder — all notebooks are in the same folder
REPO_DIR = '.'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working directory:', os.path.abspath('.'))
print('Files found:')
for f in sorted(os.listdir(REPO_DIR)):
    if f.endswith('.ipynb') or f.endswith('.py') or f.endswith('.pkl'):
        print(' ', f)


Working directory: c:\Users\joelk\OneDrive\Desktop\COMP8420 Major Project _Arti_P2
Files found:
  P1_LoRA_Complete_.ipynb
  P1_LoRA_Qwen.ipynb
  P2.ipynb
  P3_Prompt_Engineering.ipynb
  P3_Prompt_Engineering_LLM_Manager_TinyLlama_LoRA_Integrated.ipynb
  P3_Qwen_LoRA.ipynb
  P4_RAG_Agent_Engineer.ipynb
  P5_interface_final.ipynb
  app.py


## 4.Notebook Loader Utility

This section defines a utility function that dynamically loads and executes code cells from other project notebooks within the current notebook environment.

The notebook loader is used to integrate previously developed project components, enabling the final system to access functions and resources without duplicating code.

The following modules are imported through the loader:

- **P2 – Natural Language Processing (NLP):**
  - Sentiment Analysis
  - Category Classification
  - Named Entity Recognition (NER)
  - Part-of-Speech (POS) Tagging

- **P3 – Response Generation:**
  - Qwen2.5-1.5B-Instruct
  - LoRA Fine-Tuned Customer Support Model
  - Response Generation Functions

- **P4 – Retrieval-Augmented Generation (RAG) and Agent System:**
  - Policy Retrieval Engine
  - Agent Decision Logic
  - Escalation Handling

By loading these notebooks dynamically, the P5 integration notebook serves as a central orchestration layer that combines all project components into a single end-to-end customer support system.



In [6]:
import nbformat, gc

def load_notebook_functions(notebook_path, exec_globals=None):
    if exec_globals is None:
        exec_globals = globals()
    if not os.path.exists(notebook_path):
        print(f'WARNING: {notebook_path} not found. Skipping.')
        return False
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)
    loaded = errors = 0
    for cell in nb.cells:
        if cell.cell_type != 'code':
            continue
        lines = [l for l in cell.source.splitlines() if not l.strip().startswith('!')]
        src = '\n'.join(lines).strip()
        if not src:
            continue
        try:
            exec(compile(src, notebook_path, 'exec'), exec_globals)
            loaded += 1
        except Exception as e:
            errors += 1
            if 'NameError' not in type(e).__name__:
                print(f'  [skip] {type(e).__name__}: {str(e)[:80]}')
    print(f'Loaded {os.path.basename(notebook_path)}: {loaded} cells ok, {errors} skipped')
    return True

print('Loader ready.')


Loader ready.


## 4.1 LoRA Adapter Verification

This section verifies that the LoRA adapter produced during P3 fine-tuning is available and ready for use. The check confirms that the adapter directory and required weight files exist, ensuring that the fine-tuned Qwen model can be successfully loaded for response generation in the integrated customer service system.

In [7]:
import os, shutil

# Rename lora-weight to lora-weights if needed
if os.path.exists('lora-weight') and not os.path.exists('lora-weights'):
    shutil.copytree('lora-weight', 'lora-weights')
    print('Renamed lora-weight to lora-weights')
elif os.path.exists('lora-weights'):
    print('lora-weights folder found — ready.')
else:
    print('WARNING: lora-weights not found!')


lora-weights folder found — ready.


## 4.2 Loading P2 — Basic NLP Analysis

This section loads the pre-trained P2 Natural Language Processing (NLP) component directly from the saved model files rather than executing the complete training pipeline again.

Loading the saved models significantly reduces execution time and memory consumption while ensuring that the same trained components developed in P2 are used during system integration.

The P2 module provides the following capabilities:

- Sentiment Analysis
- Category Classification
- Named Entity Recognition (NER)
- Part-of-Speech (POS) Tagging

These NLP functions are used throughout the integrated customer service system to analyse customer messages, extract relevant information, and provide structured inputs to the downstream RAG and response generation components.

In [8]:
# Load only analyze() from P2 — skip training cells
import joblib, spacy, re
import numpy as np

# Load pre-trained models directly
sentiment_model  = joblib.load('models/sentiment_model.pkl')
category_model   = joblib.load('models/category_model.pkl')
nlp              = spacy.load('en_core_web_sm')

print('Models loaded directly from pkl files.')
print('Defining analyze() function...')

def analyze(text):
    # Preprocessing
    clean = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).strip()
    
    # Sentiment
    sen_result = sentiment_model.predict([clean])[0]
    sen_proba  = sentiment_model.predict_proba([clean]).max()
    
    # Category  
    cat_result = category_model.predict([clean])[0]
    
    # NER
    doc = nlp(text)
    entities = {
    'customer_name': [], 'product': [], 'shop': [],
    'date': [], 'money': [], 'issue': [],
    'order_number': [],
    'other': []
}
    for ent in doc.ents:
        if ent.label_ in ['PERSON']:
            entities['customer_name'].append(ent.text)
        elif ent.label_ in ['PRODUCT', 'ORG']:
            entities['product'].append(ent.text.lower())
        elif ent.label_ in ['GPE', 'FAC']:
            entities['shop'].append(ent.text)
        elif ent.label_ in ['DATE', 'TIME']:
            entities['date'].append(ent.text)
        elif ent.label_ in ['MONEY']:
            entities['money'].append(ent.text)

    # Extract order number manually because spaCy may not detect it
    order_matches = re.findall(
        r"(?:order\s*(?:number|no|id)?\s*(?:is|:|#)?\s*)([A-Za-z0-9\-]+)",
        text,
        flags=re.IGNORECASE
        )

    for order_id in order_matches:
        if order_id not in entities["order_number"]:
            entities["order_number"].append(order_id)
    # POS tags
    pos_tags = [(token.text, token.pos_) for token in doc]

    return {
        'clean_text': clean,
        'tokens':     clean.split(),
        'pos_tags':   pos_tags,
        'entities':   entities,
        'sentiment':  {'sentiment': sen_result, 'confidence': f'{sen_proba*100:.1f}%'},
        'category':   {'category': cat_result,  'confidence': 'keyword-based'}
    }

# Test
_t = analyze('I want to cancel my order')
print('OK — category:', _t['category'], '| sentiment:', _t['sentiment'])


Models loaded directly from pkl files.
Defining analyze() function...
OK — category: {'category': np.str_('ORDER'), 'confidence': 'keyword-based'} | sentiment: {'sentiment': 'negative', 'confidence': '97.5%'}


## 4.3 Loading P4 — RAG and Agent System

This section loads the Retrieval-Augmented Generation (RAG) and agent-based decision-making components developed in P4. The module retrieves relevant company policies, support knowledge, and determines whether a customer request should be escalated to a human support specialist.

In [9]:
P4_PATH = os.path.join(REPO_DIR, 'P4_RAG_Agent_Engineer.ipynb')
print('Loading P4 (may take 1-2 mins)...')
load_notebook_functions(P4_PATH)

# =================================
# P4 Compatibility Fix — AFTER loading P4
# =================================
from sentence_transformers import SentenceTransformer

if "embedding_info" not in globals() or not isinstance(embedding_info, dict):
    embedding_info = {}

if "embedding_model" not in globals():
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding_info.setdefault("method", "sentence-transformers")
embedding_info.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
embedding_info.setdefault("embedding_dim", 384)
embedding_info.setdefault("index_type", "FAISS")
embedding_info.setdefault("status", "loaded")
embedding_info.setdefault("vectorizer", embedding_model)

print("embedding_info ready:", embedding_info.keys())

print('Testing run_part4_rag_agent()...')
try:
    _t4 = run_part4_rag_agent(
        'My order has not arrived.',
        {'sentiment': 'negative', 'category': 'DELIVERY', 'entities': {}}
    )
    print('OK — action:', _t4['agent_output']['action'])
except Exception as e:
    print('FAILED:', e)


Loading P4 (may take 1-2 mins)...
FAISS imported successfully.
SentenceTransformer imported successfully.
datasets imported successfully.
Dataset loaded successfully.
Dataset shape: (5374, 5)
Columns: ['flags', 'instruction', 'category', 'intent', 'response']
Dataset-based FAQ entries: 48
Manual policy entries: 8
Total knowledge base entries: 56

Source type counts:
source_type
dataset_faq      48
manual_policy     8
Name: count, dtype: int64
Total RAG documents: 56

Example searchable document:

Title: ORDER - cancel_order. Source type: dataset_faq. Category: ORDER. Intent: cancel_order. Question: I can't pay for this item, cancel order {{Order Number}}. Answer: I understand the urgency and frustration of not being able to pay for this item and the need to cancel order {{Order Number}}. We're here to assist you in resolving this issue swiftly. To cancel your order, please follow these steps:

1. Log into your {{Online Company Portal Info}} using your credentials.
2. Navigate to the '{

Batches: 100%|██████████| 2/2 [00:00<00:00,  2.73it/s]


Embedding method: SentenceTransformer all-MiniLM-L6-v2
Embedding shape: (56, 384)
FAISS index created successfully.
Vector dimension: 384
Number of vectors stored: 56
Retrieval accuracy: 80.00%
Escalation accuracy: 80.00%
Average Part 4 response time: 0.0150 seconds
Agent action: ESCALATE_TO_HUMAN
Escalate: True

Retrieved context for Part 3:

[Context 1: Refund Policy | Category: REFUND | Intent: refund_policy | Score: 0.430]
Guidance: For refund requests, ask for the order number and the reason for the refund. If the product is damaged, ask for a photo. Do not guarantee a refund before review.

[Context 2: ORDER - cancel_order | Category: ORDER | Intent: cancel_order | Score: 0.345]
Guidance: For order cancellation requests, ask the customer for their order number. Explain that cancellation depends on whether the order has already been processed or shipped. If the order has already shipped, guide the customer to return or refund options.

[Context 3: Damaged Product Policy | Category

## 4.4 Loading P3 — Qwen + LoRA Response Generation

This section loads the fine-tuned Qwen2.5-1.5B-Instruct model together with the LoRA adapter produced during P3. The model is responsible for generating professional customer support responses based on the analysed customer message and retrieved support policies.

In [10]:
P3_PATH = os.path.join(REPO_DIR, 'P3_Prompt_Engineering.ipynb')
print('Loading P3 — Qwen with LoRA (please wait)...')
load_notebook_functions(P3_PATH)
print('P3 loaded.')


Loading P3 — Qwen with LoRA (please wait)...
LLM backend: Qwen/Qwen2.5-1.5B-Instruct
NVIDIA CUDA GPU acceleration available
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
LoRA adapter found: ./lora-weights
Customer message:
Hi my name is John, I ordered a MacBook Pro from Apple 3 days ago but it arrived damaged and I was charged $200 extra

Person 2 analysis:

Relevant company support policy:

1. Billing and invoice issue:
If a customer reports an incorrect charge, extra charge, or unrecognised payment,
the support assistant should apologise, ask for the order number, and request
a receipt or screenshot of the billing issue. The case should be reviewed by
the billing support team.

2. Damaged product issue:
If a product arrives damaged, the customer should be asked to provide the order
number and a photo of the damaged item. The support team can review the case
and provide the next steps for replacement, return, or refund options.

3. Escalation rule:
If the case involves both a damaged produ

W0605 10:04:56.344000 12536 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


LLM backend ready: Qwen2.5-1.5B+ LoRA adapter


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


PROMPT TYPE: Basic Prompt
STATUS: Success
RESPONSE TIME: 36.01 seconds

Dear John,

I apologize for any inconvenience you may have experienced with your recently delivered MacBook Pro. We take the integrity of our products very seriously, and we deeply regret that this damage occurred.

To address this matter promptly, please provide me with the serial number or order details associated with your MacBook Pro. This information will allow us to investigate the issue thoroughly and ensure that you receive appropriate compensation or replacement services as per Apple's policies.

We value your satisfaction and appreciate your patience while we work on resolving this situation. If there's anything else I can assist you with, please let me know. Rest assured, we're committed to making things right.

Thank you for bringing this to our attention. Let's move forward together towards finding a solution that meets both your needs and expectations.

Best regards,
Customer Support Team

PROMPT TYPE

## 5. P5 Chatbot Integration Function

This section implements the main `chatbot()` function, which serves as the central orchestration layer of the intelligent customer service system.

The function integrates all major project components and coordinates the complete end-to-end workflow from customer message processing to response generation.

### System Workflow

1. The customer message is received through the user interface.
2. The P2 NLP module performs sentiment analysis, category classification, named entity recognition (NER), and part-of-speech (POS) tagging.
3. The processed NLP output is formatted and passed to the P4 module.
4. The P4 RAG and agent system retrieves relevant company policies and determines the appropriate support action, including escalation requirements.
5. If escalation is required, the system generates a human-support escalation response.
6. If escalation is not required, the P3 Qwen + LoRA model generates a customer support response using the retrieved context and NLP analysis.
7. The final response and supporting analytics are returned to the user interface for display.

The `chatbot()` function represents the primary P5 contribution as it integrates the P2, P3, and P4 components into a unified intelligent customer service system capable of processing customer requests in real time.


In [11]:
def chatbot(customer_message):

    result = {
        "reply": "",
        "category": "UNKNOWN",
        "sentiment": "neutral",
        "escalate": False,
        "agent_action": "UNKNOWN",
        "entities": {},
        "pos_tags": [],
        "retrieved_context_text": "",
        "error": None
    }

    try:
        # ==================================================
        # P2
        # ==================================================
        p2_result = analyze(customer_message)

        category = p2_result.get("category", {})
        sentiment = p2_result.get("sentiment", {})

        result["category"] = (
            category.get("category", "UNKNOWN")
            if isinstance(category, dict)
            else str(category)
        )

        result["sentiment"] = (
            sentiment.get("sentiment", "neutral")
            if isinstance(sentiment, dict)
            else str(sentiment)
        )

        result["entities"] = p2_result.get("entities", {})
        result["pos_tags"] = p2_result.get("pos_tags", [])

        # ==================================================
        # P4
        # ==================================================
        p4_input = {
            "category": result["category"],
            "sentiment": result["sentiment"],
            "entities": result["entities"]
        }

        p4_result = run_part4_rag_agent(
            customer_message,
            p4_input,
            top_k = 2
        )

        result["agent_action"] = (
            p4_result.get("agent_output", {})
                     .get("action", "UNKNOWN")
        )

        result["escalate"] = (
            p4_result.get("escalation_output", {})
                     .get("escalate", False)
        )

        result["retrieved_context_text"] = (
            p4_result.get("retrieved_context_text", "")
        )

        # ==================================================
        # P3
        # ==================================================
        p3_result = final_response_for_UI(
            customer_message,
            p2_result,
            p4_result
        )

        if isinstance(p3_result, dict):
            result["reply"] = p3_result.get(
                "reply",
                "No reply generated."
            )
        else:
            result["reply"] = str(p3_result)

        return result

    except Exception as e:

        result["error"] = str(e)

        result["reply"] = (
            f"SYSTEM ERROR:\n{str(e)}"
        )

        return result

## SQLite Conversation History

This section creates and manages a SQLite database used to store customer interactions generated during system operation.

The database provides a lightweight and efficient mechanism for recording conversation data, enabling the system to maintain a history of customer requests and generated responses.



In [12]:
import sqlite3, json
from datetime import datetime

DB_PATH = 'conversation_history.db'

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS conversations (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp    TEXT,
            user_message TEXT,
            bot_reply    TEXT,
            category     TEXT,
            sentiment    TEXT,
            escalated    INTEGER,
            agent_action TEXT,
            entities     TEXT
        )
    ''')
    conn.commit()
    conn.close()
    print('SQLite DB ready at', DB_PATH)

def save_conversation(user_message, result):
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        'INSERT INTO conversations VALUES (NULL,?,?,?,?,?,?,?,?)',
        (datetime.now().isoformat(timespec='seconds'),
         user_message, result.get('reply',''),
         result.get('category',''), result.get('sentiment',''),
         int(result.get('escalate', False)),
         result.get('agent_action',''),
         json.dumps(result.get('entities', {})))
    )
    conn.commit()
    conn.close()

init_db()


SQLite DB ready at conversation_history.db


## End-to-End Pipeline Testing

This section evaluates the complete integrated customer service system using a collection of representative customer support scenarios.

The purpose of the testing process is to verify that all project components operate correctly when combined into a single end-to-end pipeline.

For each test case, the system performs the following steps:

1. Processes the customer message using the P2 NLP module.
2. Extracts sentiment, category, entities, and linguistic information.
3. Retrieves relevant support policies and knowledge through the P4 RAG module.
4. Determines the appropriate agent action and escalation requirements.
5. Generates a customer support response using the P3 Qwen + LoRA model.
6. Returns the final response and analytics through the P5 integrated interface.

The testing results are used to validate system functionality, response quality, policy retrieval accuracy, escalation handling, and overall integration performance. Successful execution demonstrates that the complete customer service workflow operates as intended across a variety of customer support scenarios.

In [13]:
import pandas as pd

test_messages = [
    'I want to cancel my order',
    'My MacBook arrived damaged and I was charged extra',
    'My package has not arrived yet',
    'I want to speak to a manager, this is unacceptable',
    'Can I get an invoice for my last purchase?'
]

print('Running pipeline tests...')
print('=' * 65)
rows = []
for msg in test_messages:
    print(f'\nQ: {msg}')
    res = chatbot(msg)
    save_conversation(msg, res)
    print(f'   Category : {res["category"]}')
    print(f'   Sentiment: {res["sentiment"]}')
    print(f'   Action   : {res["agent_action"]}')
    print(f'   Escalate : {res["escalate"]}')
    print(f'   Reply    : {res["reply"][:80]}...')
    if res['error']:
        print(f'   ERROR    : {res["error"]}')
    rows.append({'message': msg[:40], 'category': res['category'],
                 'sentiment': res['sentiment'], 'action': res['agent_action'],
                 'escalate': res['escalate'], 'error': res['error'] or 'None'})

print('\n' + '=' * 65)
pd.DataFrame(rows)


Running pipeline tests...

Q: I want to cancel my order
   Category : ORDER
   Sentiment: negative
   Action   : ESCALATE_TO_HUMAN
   Escalate : True
   Reply    : Dear customer,

We appreciate you reaching out about wanting to cancel your orde...

Q: My MacBook arrived damaged and I was charged extra
   Category : ORDER
   Sentiment: negative
   Action   : ESCALATE_TO_HUMAN
   Escalate : True
   Reply    : Dear customer,

I apologize for any inconvenience you've experienced with your d...

Q: My package has not arrived yet
   Category : ORDER
   Sentiment: positive
   Action   : RETRIEVE_THEN_RESPOND
   Escalate : False
   Reply    : Dear customer,

I understand that your package hasn't arrived yet. Could you kin...

Q: I want to speak to a manager, this is unacceptable
   Category : ORDER
   Sentiment: negative
   Action   : ESCALATE_TO_HUMAN
   Escalate : True
   Reply    : Dear customer,

I apologize for any inconvenience caused by our system's error. ...

Q: Can I get an invoice f

,message,category,sentiment,action,escalate,error
0,I want to cancel my order,ORDER,negative,ESCALATE_TO_HUMAN,True,None
1,My MacBook arrived damaged and I was cha,ORDER,negative,ESCALATE_TO_HUMAN,True,None
2,My package has not arrived yet,ORDER,positive,RETRIEVE_THEN_RESPOND,False,None
3,"I want to speak to a manager, this is un",ORDER,negative,ESCALATE_TO_HUMAN,True,None
4,Can I get an invoice for my last purchas,INVOICE,positive,RETRIEVE_THEN_RESPOND,False,None


## 6. Streamlit User Interface

This section generates the final `app.py` file used to deploy the integrated customer service system through a Streamlit web application.

The Streamlit interface provides a user-friendly environment that allows customers to interact with the system in a realistic web-based chat setting. It serves as the presentation layer of the project and connects directly to the integrated P2, P3, and P4 components.

The interface provides the following functionality:

- Real-time customer message submission.
- NLP analysis and entity extraction display.
- Retrieval of relevant company policies and support information.
- Agent action and escalation status monitoring.
- AI-generated customer support responses using the fine-tuned Qwen + LoRA model.
- Conversation history tracking through SQLite database integration.
- Interactive visualisation of system outputs and analytics.


In [14]:
%%writefile app.py
import streamlit as st
import sqlite3
import json
import sys
import os
import re
from datetime import datetime

# ==================================================================
# Page config
# ==================================================================
st.set_page_config(
    page_title="Customer Service AI",
    page_icon="💬",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ==================================================================
# Styles
# ==================================================================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
html, body, [class*="css"] { font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important; }
#MainMenu, footer, header { visibility: hidden; }
.stDeployButton { display: none; }
.stApp { background-color: #0f0f0f; color: #e5e5e5; }

[data-testid="stSidebar"] { background-color: #111111 !important; border-right: 1px solid #222 !important; }
[data-testid="stSidebar"] * { color: #e5e5e5 !important; }
[data-testid="stSidebar"] .stButton button { background: transparent !important; border: 1px solid #222 !important; text-align: left !important; padding: 9px 12px !important; border-radius: 8px !important; font-size: 13px !important; color: #b0b0b0 !important; width: 100% !important; }
[data-testid="stSidebar"] .stButton button:hover { background: #1e1e1e !important; color: #fff !important; border-color: #10a37f !important; }

[data-testid="stChatInput"] textarea { background-color: #1a1a1a !important; border: 1px solid #2a2a2a !important; border-radius: 24px !important; color: #e5e5e5 !important; font-size: 14px !important; }
[data-testid="stChatInput"] textarea:focus { border-color: #10a37f !important; box-shadow: 0 0 0 2px rgba(16,163,127,0.15) !important; }
[data-testid="stExpander"] { background: #1a1a1a !important; border: 1px solid #2a2a2a !important; border-radius: 8px !important; }

.section-label { font-size: 11px; color: #555; text-transform: uppercase; letter-spacing: .06em; margin: 14px 4px 6px; }
.status-dot { display:inline-block; width:7px; height:7px; border-radius:50%; margin-right:6px; }
.dot-green { background:#10a37f; } .dot-red { background:#ef4444; }

.bubble-row-user { display:flex; justify-content:flex-end; margin:10px 0; }
.bubble-row-bot  { display:flex; justify-content:flex-start; margin:10px 0; }
.bubble-user { background:#1e293b; border:1px solid #334155; color:#e5e5e5; border-radius:16px 16px 4px 16px; padding:11px 16px; max-width:74%; font-size:14px; line-height:1.6; }
.bubble-bot  { background:linear-gradient(135deg,rgba(16,163,127,.14),rgba(14,165,233,.12)); border:1px solid rgba(16,163,127,.3); color:#e5e5e5; border-radius:16px 16px 16px 4px; padding:11px 16px; max-width:74%; font-size:14px; line-height:1.6; white-space:pre-wrap; }
.bubble-err  { background:rgba(239,68,68,.1); border:1px solid rgba(239,68,68,.35); color:#fca5a5; border-radius:12px; padding:11px 16px; max-width:90%; font-size:13px; white-space:pre-wrap; font-family:monospace; }
.meta-time { font-size:10px; color:#444; margin:2px 6px; }

.info-row { display:flex; justify-content:space-between; padding:6px 0; border-bottom:1px solid #1e1e1e; font-size:12px; }
.info-label { color:#666; } .info-value { color:#e5e5e5; font-weight:500; }
.tag { display:inline-block; padding:3px 10px; border-radius:99px; font-size:11px; font-weight:500; margin:2px 3px 2px 0; }
.tag-green { background:rgba(16,163,127,.15); color:#10a37f; border:1px solid rgba(16,163,127,.25); }
.tag-red   { background:rgba(239,68,68,.12);  color:#f87171; border:1px solid rgba(239,68,68,.25); }
.tag-yellow{ background:rgba(234,179,8,.12);  color:#eab308; border:1px solid rgba(234,179,8,.25); }
.tag-blue  { background:rgba(96,165,250,.12); color:#60a5fa; border:1px solid rgba(96,165,250,.25); }
.tag-gray  { background:rgba(120,120,120,.15);color:#999;    border:1px solid rgba(120,120,120,.25); }
</style>
""", unsafe_allow_html=True)

# ==================================================================
# Paths  (must match the P5 notebook)
# ==================================================================
REPO_DIR = "."
DB_PATH  = "conversation_history.db"
P4_PATH  = os.path.join(REPO_DIR, "P4_RAG_Agent_Engineer.ipynb")
P3_PATH  = os.path.join(REPO_DIR, "P3_Qwen_LoRA.ipynb")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


# ==================================================================
# Notebook loader — execs code cells from another notebook into a globals dict
# ==================================================================
def load_notebook_functions(notebook_path, g):
    import nbformat
    if not os.path.exists(notebook_path):
        return False
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        # drop shell (!) lines and notebook magics (%)
        lines = [l for l in cell.source.splitlines()
                 if not l.strip().startswith("!") and not l.strip().startswith("%")]
        src = "\n".join(lines).strip()
        if not src:
            continue
        try:
            exec(compile(src, notebook_path, "exec"), g)
        except Exception:
            # training / NameError cells are expected to fail when loaded in isolation
            pass
    return True


def fix_lora_config():
    """Make the LoRA adapter_config compatible with the installed PEFT version."""
    import shutil
    if os.path.exists("lora-weight") and not os.path.exists("lora-weights"):
        try:
            shutil.copytree("lora-weight", "lora-weights")
        except Exception:
            pass
    cfg_path = "./lora-weights/adapter_config.json"
    if not os.path.exists(cfg_path):
        return
    try:
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        bad_keys = [
            "lora_ga_config", "alora_invocation_tokens", "arrow_config", "corda_config",
            "ensure_weight_tying", "eva_config", "exclude_modules", "qalora_group_size",
            "target_parameters", "trainable_token_indices", "use_bdlora", "use_qalora", "lora_bias",
        ]
        for k in bad_keys:
            cfg.pop(k, None)
        cfg["peft_version"] = "0.13.2"
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
    except Exception:
        pass


# ==================================================================
# Load the whole pipeline ONCE (cached for the session)
# ==================================================================
@st.cache_resource(show_spinner="Loading AI models (P2 \u2192 P4 \u2192 P3)\u2026 first run can take a few minutes.")
def load_pipeline():
    g = {}

    # ---- P2: load pkl models + spaCy, define analyze() ----
    analyze = None
    try:
        import joblib, spacy
        sentiment_model = joblib.load("models/sentiment_model.pkl")
        category_model  = joblib.load("models/category_model.pkl")
        nlp = spacy.load("en_core_web_sm")

        def analyze(text):
            clean = re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower()).strip()
            sen_result = sentiment_model.predict([clean])[0]
            sen_proba  = sentiment_model.predict_proba([clean]).max()
            cat_result = category_model.predict([clean])[0]
            doc = nlp(text)
            entities = {"customer_name": [], "product": [], "shop": [],
                        "date": [], "money": [], "issue": [], "other": []}
            for ent in doc.ents:
                if ent.label_ == "PERSON":
                    entities["customer_name"].append(ent.text)
                elif ent.label_ in ("PRODUCT", "ORG"):
                    entities["product"].append(ent.text.lower())
                elif ent.label_ in ("GPE", "FAC"):
                    entities["shop"].append(ent.text)
                elif ent.label_ in ("DATE", "TIME"):
                    entities["date"].append(ent.text)
                elif ent.label_ == "MONEY":
                    entities["money"].append(ent.text)
            pos_tags = [(t.text, t.pos_) for t in doc]
            return {
                "clean_text": clean,
                "tokens": clean.split(),
                "pos_tags": pos_tags,
                "entities": entities,
                "sentiment": {"sentiment": sen_result, "confidence": f"{sen_proba*100:.1f}%"},
                "category": {"category": cat_result, "confidence": "model"},
            }
    except Exception as e:
        print("P2 load failed:", e)

    # ---- P4: load notebook, then apply embedding compatibility fix ----
    load_notebook_functions(P4_PATH, g)
    try:
        from sentence_transformers import SentenceTransformer
        if "embedding_info" not in g or not isinstance(g.get("embedding_info"), dict):
            g["embedding_info"] = {}
        if "embedding_model" not in g:
            g["embedding_model"] = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        ei = g["embedding_info"]
        ei.setdefault("method", "sentence-transformers")
        ei.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
        ei.setdefault("embedding_dim", 384)
        ei.setdefault("index_type", "FAISS")
        ei.setdefault("status", "loaded")
        ei.setdefault("vectorizer", g["embedding_model"])
    except Exception as e:
        print("P4 embedding fix failed:", e)

    # ---- P3: fix LoRA config, then load notebook ----
    fix_lora_config()
    load_notebook_functions(P3_PATH, g)

    return {
        "analyze": analyze,
        "run_part4_rag_agent": g.get("run_part4_rag_agent"),
        "final_response_for_UI": g.get("final_response_for_UI"),
    }


PIPE = load_pipeline()
_analyze = PIPE["analyze"]
_rag     = PIPE["run_part4_rag_agent"]
_reply   = PIPE["final_response_for_UI"]


# ==================================================================
# Database
# ==================================================================
def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""CREATE TABLE IF NOT EXISTS conversations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, user_message TEXT, bot_reply TEXT,
        category TEXT, sentiment TEXT, escalated INTEGER,
        agent_action TEXT, entities TEXT)""")
    conn.commit(); conn.close()


def save_conversation(msg, res):
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT INTO conversations VALUES (NULL,?,?,?,?,?,?,?,?)",
                 (datetime.now().isoformat(timespec="seconds"), msg,
                  res.get("reply", ""), res.get("category", ""), res.get("sentiment", ""),
                  int(res.get("escalate", False)), res.get("agent_action", ""),
                  json.dumps(res.get("entities", {}))))
    conn.commit(); conn.close()


def load_history(limit=6):
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        "SELECT timestamp, user_message, category, sentiment, escalated "
        "FROM conversations ORDER BY id DESC LIMIT ?", (limit,)).fetchall()
    conn.close()
    return rows


init_db()


# ==================================================================
# Pipeline  -  P2 -> P4 -> P3.  Bot reply = P3 output ONLY. No fallback.
# ==================================================================
def run_pipeline(msg):
    res = {"reply": "", "category": "UNKNOWN", "sentiment": "neutral",
           "escalate": False, "agent_action": "UNKNOWN", "entities": {},
           "pos_tags": [], "retrieved_context_text": "", "error": None}

    # P2
    p2 = _analyze(msg)
    cat = p2.get("category", {})
    sen = p2.get("sentiment", {})
    res["category"]  = cat.get("category", "UNKNOWN") if isinstance(cat, dict) else str(cat)
    res["sentiment"] = sen.get("sentiment", "neutral") if isinstance(sen, dict) else str(sen)
    res["entities"]  = p2.get("entities", {}) or {}
    res["pos_tags"]  = p2.get("pos_tags", [])

    # P4
    p4 = _rag(msg, {"category": res["category"], "sentiment": res["sentiment"],
                    "entities": res["entities"]})
    res["agent_action"] = p4.get("agent_output", {}).get("action", "UNKNOWN")
    res["escalate"]     = bool(p4.get("escalation_output", {}).get("escalate", False))
    res["retrieved_context_text"] = p4.get("retrieved_context_text", "")

    # P3  - take the model's reply exactly as produced
    p3 = _reply(msg, p2, p4)
    res["reply"] = (p3.get("reply", "") if isinstance(p3, dict) else str(p3)).strip()
    return res


# ==================================================================
# Session state
# ==================================================================
st.session_state.setdefault("messages", [])
st.session_state.setdefault("last_res", None)
st.session_state.setdefault("customer_name", "Customer")
st.session_state.setdefault("quick_msg", None)
st.session_state.setdefault("conv_id", f"CONV-{datetime.now().strftime('%H%M%S')}")


def reset_chat():
    st.session_state.messages = []
    st.session_state.last_res = None
    st.session_state.conv_id = f"CONV-{datetime.now().strftime('%H%M%S')}"


# ==================================================================
# Left panel (replaces Streamlit sidebar so it is always visible)
# ==================================================================
def render_left_panel():
    st.markdown("""
    <div style="padding:18px 14px 14px;border-bottom:1px solid #222;">
        <div style="display:flex;align-items:center;gap:10px;">
            <div style="width:32px;height:32px;background:linear-gradient(135deg,#10a37f,#0ea5e9);
                        border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:16px;">💬</div>
            <div>
                <div style="font-size:14px;font-weight:600;">ServiceAI</div>
                <div style="font-size:11px;color:#555;">COMP8420 · Use Case 1</div>
            </div>
        </div>
    </div>""", unsafe_allow_html=True)

    if st.button("\uff0b  New conversation", key="new_conv", use_container_width=True):
        reset_chat(); st.rerun()

    st.markdown('<div class="section-label">Customer</div>', unsafe_allow_html=True)
    name = st.text_input("name", value=st.session_state.customer_name,
                         placeholder="Customer name\u2026", label_visibility="collapsed")
    if name != st.session_state.customer_name:
        st.session_state.customer_name = name

    st.markdown('<div class="section-label">Quick Tests</div>', unsafe_allow_html=True)
    quick = [("🛍️", "I want to cancel my order"),
             ("📦", "My package has not arrived yet"),
             ("💳", "I was charged extra on my invoice"),
             ("😡", "I want to speak to a manager, this is unacceptable"),
             ("📦", "Hi, my name is John. I ordered a MacBook Pro from Apple last week. It arrived damaged and I was charged an extra $200.")]
    for icon, label in quick:
        if st.button(f"{icon}  {label}", key=f"q_{label}", use_container_width=True):
            st.session_state.quick_msg = label
            st.rerun()

    st.markdown('<div class="section-label">Pipeline Status</div>', unsafe_allow_html=True)
    for lbl, ok in [("P2 NLP Analysis", _analyze is not None),
                    ("P4 RAG + Agent", _rag is not None),
                    ("P3 LLM Response", _reply is not None)]:
        dot = "dot-green" if ok else "dot-red"
        st.markdown(f'<div style="font-size:12px;padding:4px;color:#aaa;">'
                    f'<span class="status-dot {dot}"></span>{lbl}</div>', unsafe_allow_html=True)

    st.markdown('<div class="section-label">Recent</div>', unsafe_allow_html=True)
    try:
        for ts, m, c, s, e in load_history(5):
            icon = "🔴" if e else "🟢"
            st.markdown(f'<div style="font-size:12px;color:#999;padding:4px 2px;">'
                        f'{icon} {c or "—"} · <span style="color:#555;">{(m or "")[:24]}\u2026</span></div>',
                        unsafe_allow_html=True)
    except Exception:
        st.markdown('<div style="font-size:12px;color:#444;padding:4px;">No history yet.</div>',
                    unsafe_allow_html=True)



# ==================================================================
# Main layout
# ==================================================================
col_left, col_chat, col_info = st.columns([1.15, 3, 2], gap="medium")

with col_left:
    render_left_panel()

with col_chat:
    top_l, _top_r = st.columns([1, 4])
    with top_l:
        if st.button("\uff0b New Chat", key="new_chat_top", use_container_width=True):
            reset_chat(); st.rerun()

    r = st.session_state.last_res
    cat_val = r.get("category", "—") if r else "—"
    sen_val = (r.get("sentiment", "neutral") if r else "neutral")
    esc_val = r.get("escalate", False) if r else False
    sen_map = {"positive": ("tag-green", "POSITIVE"),
               "negative": ("tag-red", "NEGATIVE"),
               "neutral":  ("tag-yellow", "NEUTRAL")}
    sen_cls, sen_lbl = sen_map.get(str(sen_val).lower(), ("tag-gray", str(sen_val).upper()))
    esc_cls, esc_lbl = ("tag-red", "Escalated") if esc_val else ("tag-green", "Automated")
    init = (st.session_state.customer_name or "C")[0].upper()

    st.markdown(f"""
    <div style="padding:14px 4px 12px;border-bottom:1px solid #1e1e1e;
                display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;">
        <div style="display:flex;align-items:center;gap:10px;">
            <div style="width:36px;height:36px;background:linear-gradient(135deg,#6366f1,#8b5cf6);
                        border-radius:50%;display:flex;align-items:center;justify-content:center;
                        font-weight:600;color:#fff;">{init}</div>
            <div>
                <div style="font-size:15px;font-weight:600;">{st.session_state.customer_name}</div>
                <div style="font-size:11px;color:#555;">{st.session_state.conv_id} · Web chat</div>
            </div>
        </div>
        <div>
            <span class="tag tag-blue">{cat_val}</span>
            <span class="tag {sen_cls}">{sen_lbl}</span>
            <span class="tag {esc_cls}">{esc_lbl}</span>
        </div>
    </div>""", unsafe_allow_html=True)

    # scrollable chat area (fixed height, scrolls internally)
    chat_box = st.container(height=440, border=False)
    with chat_box:
        if not st.session_state.messages:
            st.markdown(f"""
            <div style="text-align:center;padding:60px 20px;color:#444;">
                <div style="font-size:40px;">👋</div>
                <div style="font-size:15px;color:#666;margin-top:8px;">New conversation with {st.session_state.customer_name}</div>
                <div style="font-size:12px;color:#444;margin-top:4px;">Type a message or use a quick test from the left panel</div>
            </div>""", unsafe_allow_html=True)
        else:
            for m in st.session_state.messages:
                content = (m["content"].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))
                t = m.get("time", "")
                if m["role"] == "user":
                    st.markdown(f'<div class="bubble-row-user"><div class="bubble-user">{content}</div></div>'
                                f'<div class="meta-time" style="text-align:right;">{t}</div>',
                                unsafe_allow_html=True)
                elif m.get("is_error"):
                    st.markdown(f'<div class="bubble-row-bot"><div class="bubble-err">{content}</div></div>',
                                unsafe_allow_html=True)
                else:
                    st.markdown(f'<div class="bubble-row-bot"><div class="bubble-bot">{content}</div></div>'
                                f'<div class="meta-time">{t}</div>', unsafe_allow_html=True)
        # auto-scroll to the latest message
        st.markdown('<div id="chat-end"></div>'
                    '<script>var e=document.getElementById("chat-end");'
                    'if(e){e.scrollIntoView();}</script>', unsafe_allow_html=True)

    # input
    user_input = st.chat_input(f"Message {st.session_state.customer_name}\u2026")
    if st.session_state.quick_msg:
        user_input = st.session_state.quick_msg
        st.session_state.quick_msg = None

    if user_input:
        st.session_state.messages.append(
            {"role": "user", "content": user_input, "time": datetime.now().strftime("%H:%M")})
        with st.spinner("Analysing\u2026"):
            try:
                res = run_pipeline(user_input)
                st.session_state.last_res = res
                save_conversation(user_input, res)
                st.session_state.messages.append(
                    {"role": "assistant", "content": res["reply"] or "(empty response from P3)",
                     "time": datetime.now().strftime("%H:%M")})
            except Exception as e:
                # surface the real error instead of a canned fallback
                st.session_state.messages.append(
                    {"role": "assistant", "content": f"PIPELINE ERROR:\n{e}",
                     "time": datetime.now().strftime("%H:%M"), "is_error": True})
        st.rerun()


# ==================================================================
# Right panel - live NLP insights (metadata only; reply stays pure P3)
# ==================================================================
with col_info:
    st.markdown('<div style="padding:14px 4px 10px;border-bottom:1px solid #1e1e1e;">'
                '<div style="font-size:14px;font-weight:600;">Insights</div>'
                '<div style="font-size:11px;color:#555;">Real-time NLP analysis</div></div>',
                unsafe_allow_html=True)

    r = st.session_state.last_res
    if r:
        for label, value in [
            ("Conversation ID", st.session_state.conv_id),
            ("Channel", "Web chat"),
            ("Status", "🔴 Escalated" if r.get("escalate") else "🟢 Open"),
            ("Category", r.get("category", "—")),
            ("Sentiment", str(r.get("sentiment", "—")).title()),
            ("Agent Action", str(r.get("agent_action", "—")).replace("_", " ").title()),
        ]:
            st.markdown(f'<div class="info-row"><span class="info-label">{label}</span>'
                        f'<span class="info-value">{value}</span></div>', unsafe_allow_html=True)

        st.markdown('<div class="section-label">Extracted Entities (NER)</div>', unsafe_allow_html=True)
        ents = r.get("entities", {})
        has = False
        for label, key in [("Customer", "customer_name"), ("Product", "product"),
                           ("Shop", "shop"), ("Date", "date"), ("Money", "money"), ("Issue", "issue")]:
            vals = ents.get(key, [])
            if vals:
                has = True
                st.markdown(f'<div class="info-row"><span class="info-label">{label}</span>'
                            f'<span class="info-value" style="color:#10a37f;">'
                            f'{", ".join(str(v) for v in vals)}</span></div>', unsafe_allow_html=True)
        if not has:
            st.markdown('<div style="font-size:12px;color:#444;padding:6px 0;">No entities extracted.</div>',
                        unsafe_allow_html=True)

        pos = r.get("pos_tags", [])
        if pos:
            import pandas as pd
            with st.expander("🏷️ POS Tags — P2"):
                st.dataframe(pd.DataFrame([{"Word": w, "POS": p} for w, p in pos[:14]]),
                             hide_index=True, use_container_width=True)

        rag = r.get("retrieved_context_text", "")
        if rag:
            with st.expander("🔍 RAG Context — P4"):
                st.markdown(f'<div style="font-size:11px;color:#aaa;line-height:1.6;white-space:pre-wrap;">'
                            f'{rag[:600]}{"…" if len(rag) > 600 else ""}</div>', unsafe_allow_html=True)
    else:
        st.markdown('<div style="text-align:center;padding:48px 16px;color:#333;">'
                    '<div style="font-size:28px;">📊</div>'
                    '<div style="font-size:13px;color:#444;margin-top:8px;">'
                    'Send a message to see real-time NLP insights.</div></div>', unsafe_allow_html=True)

    st.markdown('<div class="section-label">History</div>', unsafe_allow_html=True)
    try:
        import pandas as pd
        rows = load_history(6)
        if rows:
            df = pd.DataFrame(rows, columns=["Time", "Message", "Category", "Sentiment", "Escalated"])
            df["Message"]   = df["Message"].str[:26] + "\u2026"
            df["Time"]      = df["Time"].str[11:16]
            df["Escalated"] = df["Escalated"].apply(lambda x: "🔴" if x else "🟢")
            st.dataframe(df, hide_index=True, use_container_width=True)
        else:
            st.markdown('<div style="font-size:12px;color:#333;">No conversations yet.</div>',
                        unsafe_allow_html=True)
    except Exception:
        st.markdown('<div style="font-size:12px;color:#333;">History unavailable.</div>',
                    unsafe_allow_html=True)

Overwriting app.py


## 7. Launching the Streamlit Application

This section launches the Streamlit web application that serves as the user interface for the integrated Intelligent Customer Service System.

Once the application is started, users can access the system through a local web browser and interact with the chatbot in a realistic customer support environment.

The deployed application integrates all major project components:

- **P2:** Natural Language Processing (NLP) analysis, including sentiment analysis, category classification, named entity recognition (NER), and part-of-speech tagging.
- **P4:** Retrieval-Augmented Generation (RAG) and agent-based decision making.
- **P3:** Qwen + LoRA response generation.
- **P5:** System integration, conversation logging, and user interface deployment.

**Note:** The generated `app.py` file contains the complete Streamlit application and can be executed directly using Streamlit without requiring the notebook to be re-run. This enables the integrated customer service system to be deployed, demonstrated, and tested independently through a web-based interface.

In [15]:
import subprocess, time

subprocess.Popen(
    ['streamlit', 'run', 'app.py'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print('Streamlit started!')
print('Open browser at: http://localhost:8501')


Streamlit started!
Open browser at: http://localhost:8501


# Self-Evaluation Report

As Person 5, my role was to integrate all project components into a complete end-to-end Intelligent Customer Service System and develop the final user interface. My work focused on connecting the NLP pipeline developed in P2, the response generation module developed in P3, and the Retrieval-Augmented Generation (RAG) and agent system developed in P4 into a unified customer support application.

The integration process involved loading and coordinating the outputs generated by each component, ensuring that customer messages flowed correctly through the NLP analysis stage, policy retrieval stage, escalation decision stage, and response generation stage. I also implemented the central `chatbot()` function, which acts as the orchestration layer of the entire system.

Besides integrating the system, I created a web app using Streamlit that simulates a real customer support experience. The app lets users send in support requests, get responses from AI, see extracted details and sentiment analysis, check the policy context that was found, and track escalation decisions as they happen in real time.

To help with checking and testing, I set up conversation logging with SQLite, which lets customer chats be saved and looked at later. I also did end-to-end testing with various customer support situations to make sure all the modules worked properly when they were put together.

The end result of my work is a complete, unified intelligent customer service app that brings together natural language processing, retrieval-augmented reasoning, escalation management, and response generation using large language models, all in one system that can be deployed.

# System Integration Justification

The main goal of the P5 component was to turn separately created modules into a single, ready-to-use customer service system. Instead of creating new machine learning models, this phase focused on system design, compatibility between systems, how the system is set up, and the overall experience for users.

The system was built in a way that allows each part to do its specific job while helping the whole process work smoothly. P2 is responsible for analyzing language and extracting key information, P4 finds the right support information and decides if the issue needs to be escalated, and P3 creates professional and helpful responses for customers. The P5 integration layer brings all these parts together and shows the final output via a user-friendly web interface.

The Streamlit framework was chosen because it allows for quick development and deployment of machine learning apps, and it offers an interactive platform that's great for showing off and testing the applications. SQLite was picked for storing conversations because it is simple to use and easy to include in a single, self-contained program.

This modular design makes it easier to maintain, scale up, and add new features in the future. Individual parts of the system can be upgraded or swapped out without needing major changes to how the entire system is organized.


# Conclusion

This project demonstrated the successful integration of multiple artificial intelligence technologies into a practical Intelligent Customer Service System. The final solution combines Natural Language Processing, Retrieval-Augmented Generation, agent-based decision making, and Large Language Model response generation to provide contextual, policy-aware customer support.

My contribution focused on connecting all project components into a single operational system and developing the deployment interface through Streamlit. The resulting application enables customers to interact with the system through a realistic web-based environment while providing transparency into the underlying NLP, retrieval, and decision-making processes.

The final workflow combines:

- Person 1's model fine-tuning and LoRA implementation.
- Person 2's NLP analysis and information extraction pipeline.
- Person 4's RAG retrieval and escalation framework.
- Person 5's System integration, deployment, database logging, and user interface development.

These parts work together to create a full smart customer service system that can understand customer questions, find the right support details, make professional replies, and help pass the case to a person if needed. The modular design also gives a solid base for future upgrades, extra support options, and using it in actual customer service settings.